# 02: Results

*Educational project, not for clinical use.* Reads finished runs (from Colab or local) and
regenerates the tables and figures. Run `scripts/aggregate_results.py` and
`scripts/analyze_run.py` first, or call them below. **Every table states its split; proxy/image-level
splits may be optimistic.**

In [ ]:
# Run from the repo root (or adjust). Uses the installed `mds` package.
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
DATA_ROOT = Path(os.environ.get("BONESCOPE_DATA", "data/raw"))
print(ROOT, DATA_ROOT, DATA_ROOT.exists())

In [ ]:
RUNS = Path(os.environ.get("BONESCOPE_RUNS", "runs"))
RUN, SPLIT = "convnext_ce", "proxy"      # edit
!python scripts/aggregate_results.py --runs {RUNS} --out reports/results.md
from IPython.display import Markdown, display
display(Markdown(Path("reports/results.md").read_text()))

In [ ]:
import pandas as pd
logs = {p.parent.name: pd.read_csv(p) for p in sorted((RUNS / RUN / SPLIT).glob("fold*/log.csv"))}
ax = None
for fold, log in logs.items():
    ax = log.plot(x="epoch", y="val_macro_f1", ax=ax, label=fold)
ax.set_ylabel("validation macro-F1"); ax.set_title(f"{RUN} learning curves")

In [ ]:
# Fill in after the class list is confirmed (Checkpoint 1)
BLAST = "<blast class>"; NON_NUC = ""
# !python scripts/analyze_run.py --run {RUNS}/{RUN}/{SPLIT} --blast-classes {BLAST} --non-nucleated {NON_NUC}
# display(Markdown(Path(f"reports/analysis_{RUN}_{SPLIT}.md").read_text()))

In [ ]:
from IPython.display import Image
for f in ["confusion", "reliability", "virtual_patients"]:
    p = Path(f"reports/figures/{f}_{RUN}_{SPLIT}.png")
    if p.exists(): display(Image(str(p), width=800))